# Lab 3: Knowledge Base Integration (20 minutes)

In this lab, you will integrate Amazon Bedrock Knowledge Base to provide your agent with authoritative AWS terminology, ensuring translation consistency and accuracy.

## Learning Objectives

**You'll Learn:**
- ✅ Deploy Bedrock Knowledge Base with CloudFormation
- ✅ Create tools that query Knowledge Base
- ✅ Integrate terminology validation into refinement loop
- ✅ Achieve production-quality translation consistency


## Step 1: Setup and Imports

In [1]:
!pip install -q -r requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sparkmagic 0.21.0 requires pandas<2.0.0,>=0.17.1, but you have pandas 2.3.3 which is incompatible.


In [2]:
from strands import Agent, tool
from strands.models import BedrockModel
import boto3
import sys
sys.path.append('lab_helpers')
from translation_tools import evaluate_translation_quality, refine_translation_section
from utils import (
    print_section_header, print_success, print_info, print_error,
    get_aws_region, get_account_id, create_s3_bucket_for_kb,
    upload_terminology_to_s3, query_knowledge_base
)

print_success("Imports complete!")
print_info(f"AWS Region: {get_aws_region()}")
print_info(f"Account ID: {get_account_id()}")

✅ Imports complete!
ℹ️  AWS Region: us-east-1
ℹ️  Account ID: 291480921170


## Step 2: Create S3 Bucket and Upload Terminology

Knowledge Base needs terminology data stored in S3.

In [4]:
print_section_header("Preparing Knowledge Base Data")

# Create S3 bucket
bucket_name = create_s3_bucket_for_kb()

# Upload terminology file
terminology_file = 'sample_data/aws_terminology.csv'
s3_uri = upload_terminology_to_s3(bucket_name, terminology_file)

print_success(f"Terminology uploaded to: {s3_uri}")


  Preparing Knowledge Base Data

✅ Created S3 bucket: translation-kb-291480921170-us-east-1
✅ Uploaded terminology to: s3://translation-kb-291480921170-us-east-1/terminology/aws_terminology.csv
✅ Terminology uploaded to: s3://translation-kb-291480921170-us-east-1/terminology/aws_terminology.csv


## Step 3: Deploy Knowledge Base with CloudFormation

We'll use CloudFormation to create the Knowledge Base infrastructure.

In [9]:
print_section_header("Deploying Knowledge Base")

# CloudFormation stack parameters
stack_name = "translation-workshop-kb"
template_file = "infrastructure/knowledge-base.yaml"

cfn_client = boto3.client('cloudformation')

# Read template
with open(template_file, 'r') as f:
    template_body = f.read()

print("📦 Creating CloudFormation stack...")
print(f"   Stack name: {stack_name}")
print(f"   S3 bucket: {bucket_name}")

try:
    response = cfn_client.create_stack(
        StackName=stack_name,
        TemplateBody=template_body,
        Parameters=[
            {
                'ParameterKey': 'S3BucketName',
                'ParameterValue': bucket_name
            }
        ],
        Capabilities=['CAPABILITY_IAM']
    )
    
    stack_id = response['StackId']
    print_success(f"Stack creation initiated: {stack_id}")
    print_info("⏳ Waiting for stack to complete (this may take 3-5 minutes)...")
    
    # Wait for stack creation
    waiter = cfn_client.get_waiter('stack_create_complete')
    waiter.wait(StackName=stack_name)
    
    print_success("Knowledge Base deployed successfully!")
    
except cfn_client.exceptions.AlreadyExistsException:
    print_info("Stack already exists, using existing resources")
except Exception as e:
    print_error(f"Error deploying stack: {str(e)}")
    raise


  Deploying Knowledge Base

📦 Creating CloudFormation stack...
   Stack name: translation-workshop-kb
   S3 bucket: translation-kb-291480921170-us-east-1
✅ Stack creation initiated: arn:aws:cloudformation:us-east-1:291480921170:stack/translation-workshop-kb/de432410-c989-11f0-b1b9-0ec7ac0af9e7
ℹ️  ⏳ Waiting for stack to complete (this may take 3-5 minutes)...
❌ Error deploying stack: Waiter StackCreateComplete failed: Waiter encountered a terminal failure state: For expression "Stacks[].StackStatus" we matched expected path: "ROLLBACK_COMPLETE" at least once


WaiterError: Waiter StackCreateComplete failed: Waiter encountered a terminal failure state: For expression "Stacks[].StackStatus" we matched expected path: "ROLLBACK_COMPLETE" at least once

## Step 4: Get Knowledge Base ID

Retrieve the Knowledge Base ID from CloudFormation outputs.

In [ ]:
print_section_header("Retrieving Knowledge Base Information")

# Get stack outputs
response = cfn_client.describe_stacks(StackName=stack_name)
outputs = response['Stacks'][0]['Outputs']

# Extract Knowledge Base ID
kb_id = None
for output in outputs:
    if output['OutputKey'] == 'KnowledgeBaseId':
        kb_id = output['OutputValue']
        break

if kb_id:
    print_success(f"Knowledge Base ID: {kb_id}")
else:
    print_error("Could not retrieve Knowledge Base ID")
    raise Exception("Knowledge Base ID not found in stack outputs")

## Step 5: Test Knowledge Base Query

Verify the Knowledge Base is working correctly.

In [ ]:
print_section_header("Testing Knowledge Base")

# Test query
test_query = "What is the correct Russian translation for 'serverless'?"

print(f"🔍 Query: {test_query}\n")

try:
    results = query_knowledge_base(kb_id, test_query, max_results=3)
    
    print("📚 Knowledge Base Results:")
    print("-" * 60)
    
    for i, result in enumerate(results.get('retrievalResults', []), 1):
        content = result.get('content', {}).get('text', 'No content')
        score = result.get('score', 0)
        print(f"\nResult {i} (Score: {score:.3f}):")
        print(content)
    
    print_success("Knowledge Base is working correctly!")
    
except Exception as e:
    print_error(f"Error querying Knowledge Base: {str(e)}")
    raise

## Step 6: Create KB Query Tool for Agent

**Core Pattern**: Tools that access external knowledge sources

In [ ]:
@tool
def query_aws_terminology(term: str) -> str:
    """
    Query AWS terminology Knowledge Base for correct Russian translations.
    
    Args:
        term: Technical term to look up (e.g., 'serverless', 'Lambda', 'scaling')
        
    Returns:
        Correct Russian translation and definition from authoritative source
    """
    try:
        query = f"What is the correct Russian translation for '{term}'?"
        results = query_knowledge_base(kb_id, query, max_results=1)
        
        if results.get('retrievalResults'):
            content = results['retrievalResults'][0].get('content', {}).get('text', '')
            return f"Terminology for '{term}': {content}"
        else:
            return f"No terminology found for '{term}' in Knowledge Base"
    except Exception as e:
        return f"Error querying terminology: {str(e)}"

print_success("KB query tool created!")
print_info("Agent can now validate terminology against authoritative source")

## Step 7: Create Production-Quality Translation Agent

Combine all tools: evaluation, refinement, and terminology validation.

In [ ]:
print_section_header("Creating Production-Quality Agent")

# Configure model
model = BedrockModel(
    model_id="us.anthropic.claude-3-7-sonnet-20250219-v1:0"
)

# Create agent with all tools
production_translator = Agent(
    model=model,
    tools=[
        evaluate_translation_quality,
        refine_translation_section,
        query_aws_terminology
    ],
    system_prompt="""
    You are a production-quality translation agent with access to authoritative AWS terminology.
    
    Your workflow:
    1. Translate text from English to Russian
    2. Identify technical terms that need validation
    3. Use query_aws_terminology tool to verify correct translations
    4. Use evaluate_translation_quality to assess overall quality
    5. If quality_score < 85 or terminology issues found:
       - Use refine_translation_section to fix issues
       - Re-validate terminology
       - Re-evaluate quality
    6. Repeat until quality_score >= 85 AND all terminology is validated
    7. Return final translation with validation summary
    
    Translation Rules:
    - ALWAYS query Knowledge Base for AWS service names and technical terms
    - Use exact terminology from Knowledge Base
    - Keep AWS service names in English unless KB says otherwise
    - Use natural, fluent Russian
    - Maintain formal tone for technical documentation
    
    Show your work:
    - List terms validated against Knowledge Base
    - Explain refinements made
    - Report final quality score and iteration count
    """
)

print_success("Production-quality translation agent ready!")
print_info("Agent has access to:")
print("   ✓ Self-evaluation capability")
print("   ✓ Iterative refinement")
print("   ✓ Authoritative terminology validation")

## Step 8: Run Production Translation

Watch the agent validate terminology and refine translation.

In [ ]:
print_section_header("Production Translation with KB Validation")

# Load sample document
with open('sample_data/aws_lambda_intro.txt', 'r', encoding='utf-8') as f:
    aws_doc_text = f.read()

first_paragraph = aws_doc_text.split('\n\n')[0]

translation_request = f"""
Translate this AWS documentation to Russian with full quality assurance:

{first_paragraph}

Requirements:
1. Validate ALL technical terms using the Knowledge Base
2. Iteratively refine until quality >= 85
3. Show which terms you validated
4. Explain your refinements
"""

print("🤖 Agent is working with KB validation...\n")
result = production_translator(translation_request)

print("\n" + "="*60)
print("📊 Production-Quality Translation:")
print("="*60)
print(result.message)

## Step 9: Compare All Three Approaches

See the evolution from Lab 1 to Lab 3.

In [ ]:
print_section_header("Evolution: Lab 1 → Lab 2 → Lab 3")

print("""
📈 Translation Quality Evolution:

Lab 1 - Basic One-Shot:
  ❌ No quality evaluation
  ❌ No refinement capability
  ❌ No terminology validation
  ⚡ Fast but inconsistent quality

Lab 2 - Self-Evaluating:
  ✅ Quality evaluation
  ✅ Iterative refinement
  ❌ No terminology validation
  ⚡ Better quality, still guessing on terms

Lab 3 - Production-Quality:
  ✅ Quality evaluation
  ✅ Iterative refinement
  ✅ Authoritative terminology validation
  ✅ Consistent, professional output
  ⚡ Slower but production-ready

💡 Key Insight: Each lab added a layer of intelligence,
   moving from simple translation to autonomous quality assurance.
""")

## Lab 3 Complete - Key Takeaways

### 🎉 What You've Achieved:

✅ **Infrastructure Skills:**
- Deployed Bedrock Knowledge Base with CloudFormation
- Configured S3 data sources
- Integrated external knowledge into agent workflows

✅ **Agentic Patterns:**
- Tools that query external knowledge sources
- Multi-tool coordination (evaluation + refinement + validation)
- Quality-driven workflows with authoritative data

✅ **Production Readiness:**
- Terminology consistency enforcement
- Authoritative source validation
- Repeatable quality standards
- Transparent reasoning and validation

**🎯 What Makes This Production-Quality:**
- Agent validates against authoritative sources
- Consistent terminology across all translations
- Self-correcting quality assurance
- Auditable decision-making process

### 🚀 Next: Lab 4 - AgentCore Deployment

In Lab 4, you'll:
- Package the agent for production deployment
- Deploy to Amazon Bedrock AgentCore Runtime
- Test the production endpoint
- Monitor with AgentCore Observability
